In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Login succeeded: staff shell is up (verified: nav[aria-label='Staff'] in AppShell.jsx)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    print("Logged in at:", driver.current_url)

    # Navigate to a real authenticated page (Dashboard nav item, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Dashboard')]"))).click()
    time.sleep(2)
    print("On page:", driver.current_url)

    # Refresh and verify the session persists (tokens live in localStorage, verified in auth.js)
    driver.refresh()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    assert not driver.find_elements(By.ID, "username"), "Session lost: login form shown after refresh."
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert token, "Access token missing after refresh."
    body = driver.find_element(By.TAG_NAME, "body").text
    assert len(body.strip()) > 200, "Authenticated page blank after refresh."
    print("After refresh:", driver.current_url)
    print("PASS: Session persistence verified")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("48_session_persistence_FAIL.png")
finally:
    driver.quit()